# Stage 12 — Banc d'essai des améliorations (CV honnête)

Ce notebook rassemble les idées qu'on a testées pour dépasser la baseline (~0.819 board, ~0.951 en CV honnête à 54%). **Toutes sont évaluées en CV honnête**, jamais sur le board.

**Conclusion d'avance (honnête) :** aucune ne dépasse le bruit (±0.009). On est proche du plafond du CSV fourni. Ce notebook sert de preuve/trace pour le rapport, et de base si tu veux continuer.

In [1]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
TRAIN_PATH="Train.csv"
NODATA=-9999
MONTHS=[f"{i:02d}" for i in range(1,13)]
BANDS=["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]
WINTER_BIAS=np.array([.88,.78,.62,.50,.41,.43,.36,.41,.50,.80,.75,.87])
def sr(a,b): return (a-b)/(a+b)

def interpolate_months(df):
    d=df.copy()
    for b in BANDS:
        cols=[f"{b}_{m}" for m in MONTHS]; M=d[cols].values.astype(float)
        for i in range(len(d)):
            row=M[i]; v=np.isfinite(row)
            if v.sum()>=2: M[i]=np.interp(np.arange(12),np.arange(12)[v],row[v])
            elif v.sum()==1: M[i]=np.where(v,row,row[v][0])
        d[cols]=M
    return d

def build_features(df, use_interp=False):
    if use_interp: df=interpolate_months(df)
    feats={}; idx={}
    for m in MONTHS:
        g,n,r,s1=df[f"green_{m}"],df[f"nir_{m}"],df[f"red_{m}"],df[f"swir1_{m}"]
        idx.setdefault("NDWI",[]).append(sr(g,n)); idx.setdefault("MNDWI",[]).append(sr(g,s1))
        idx.setdefault("NDVI",[]).append(sr(n,r)); idx.setdefault("NDMI",[]).append(sr(n,s1))
        idx.setdefault("VHmVV",[]).append(df[f"VH_{m}"]-df[f"VV_{m}"])
    def s(mat,nm):
        a=np.column_stack([np.asarray(x,float) for x in mat])
        feats[f"{nm}_mean"]=np.nanmean(a,1);feats[f"{nm}_std"]=np.nanstd(a,1);feats[f"{nm}_min"]=np.nanmin(a,1)
        feats[f"{nm}_max"]=np.nanmax(a,1);feats[f"{nm}_range"]=np.nanmax(a,1)-np.nanmin(a,1);feats[f"{nm}_med"]=np.nanmedian(a,1)
        feats[f"{nm}_nvalid"]=np.isfinite(a).sum(1)
    for b in BANDS: s([df[f"{b}_{m}"] for m in MONTHS],b)
    for k,v in idx.items(): s(v,k)
    return pd.DataFrame(feats,index=df.index).replace([np.inf,-np.inf],np.nan)

def corrupt(df,seed=0):
    rng=np.random.RandomState(seed);d=df.copy();cols={b:[f"{b}_{m}" for m in MONTHS] for b in BANDS}
    arr={b:d[cols[b]].values.astype(float) for b in BANDS}
    for i in range(len(d)):
        miss=rng.rand(12)<WINTER_BIAS*rng.uniform(0.7,1.1)
        for b in BANDS: arr[b][i,miss]=np.nan
    for b in BANDS: d[cols[b]]=arr[b]
    return d

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

train=pd.read_csv(TRAIN_PATH).replace(NODATA,np.nan); y=train["label"].values
Xref=build_features(train)
clusters=KMeans(10,random_state=42,n_init=10).fit_predict(Xref.fillna(Xref.median()))
gkf=GroupKFold(5)

def evaluate(model_fn, use_interp=False, n_draws=300):
    """CV honnête: cluster-holdout + validation corrompue, évaluée dans le monde à 54%."""
    oof=np.zeros(len(y))
    for tr,va in gkf.split(Xref,y,clusters):
        parts=[train.iloc[tr]]+[corrupt(train.iloc[tr],seed=i+1) for i in range(2)]
        aug=pd.concat(parts,ignore_index=True); ya=np.concatenate([y[tr]]*3)
        Xa=build_features(aug,use_interp); Xv=build_features(corrupt(train.iloc[va],seed=99),use_interp)
        oof[va]=model_fn(Xa,ya,Xv)
    rng=np.random.RandomState(1); pos=np.where(y==1)[0]; neg=np.where(y==0)[0]; sc=[]
    for _ in range(n_draws):
        idx=np.concatenate([rng.choice(pos,int(400*.544),replace=True),rng.choice(neg,400-int(400*.544),replace=True)])
        yy=y[idx];pp=oof[idx];pred=(pp>=.5).astype(int); sc.append(0.6*f1_score(yy,pred)+0.4*roc_auc_score(yy,pp))
    return np.mean(sc),np.std(sc)

## Les modèles testés

In [3]:
def m_base(Xa,ya,Xv):
    h=HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=7,l2_regularization=1.0,random_state=42).fit(Xa,ya)
    l=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),LogisticRegression(max_iter=2000)).fit(Xa,ya)
    return 0.5*h.predict_proba(Xv)[:,1]+0.5*l.predict_proba(Xv)[:,1]

def m_ens3(Xa,ya,Xv):
    h1=HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=7,l2_regularization=1.0,random_state=42).fit(Xa,ya)
    h2=HistGradientBoostingClassifier(max_iter=600,learning_rate=0.03,max_leaf_nodes=31,l2_regularization=2.0,random_state=7).fit(Xa,ya)
    l=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),LogisticRegression(max_iter=2000)).fit(Xa,ya)
    return (h1.predict_proba(Xv)[:,1]+h2.predict_proba(Xv)[:,1]+l.predict_proba(Xv)[:,1])/3

print("Résultats (CV honnête, monde à 54% de bassins):\n")
mb,sb=evaluate(m_base, use_interp=False);  print(f"  baseline (Stage10)       : {mb:.4f} ± {sb:.4f}  [référence]")
mi,si=evaluate(m_base, use_interp=True);   print(f"  + interpolation          : {mi:.4f}  diff {mi-mb:+.4f}")
me,se=evaluate(m_ens3, use_interp=False);  print(f"  + ensemble 3 modèles     : {me:.4f}  diff {me-mb:+.4f}")
mc,sc=evaluate(m_ens3, use_interp=True);   print(f"  + COMBO (interp+ens3)    : {mc:.4f}  diff {mc-mb:+.4f}")
print(f"\nSeuil de bruit ≈ ±{sb:.3f}. Un gain n'est RÉEL que s'il dépasse ce seuil.")

Résultats (CV honnête, monde à 54% de bassins):

  baseline (Stage10)       : 0.9487 ± 0.0091  [référence]
  + interpolation          : 0.9550  diff +0.0063
  + ensemble 3 modèles     : 0.9539  diff +0.0052
  + COMBO (interp+ens3)    : 0.9573  diff +0.0086

Seuil de bruit ≈ ±0.009. Un gain n'est RÉEL que s'il dépasse ce seuil.


## À retenir
- Tous les leviers testés donnent un gain **positif mais sous le bruit** (<0.009). Rien de prouvé.
- Le cumul (combo) ne s'additionne pas vraiment → les idées capturent un signal qui se recoupe.
- **Diagnostic : on est proche du plafond du CSV fourni.** Le vrai levier pour rejoindre les ~0.91 est probablement **de meilleures données** (Google Earth Engine), pas un meilleur modèle.